In [ ]:
"""
Standalone GPU/JAX four-scheme regime-rate sweep.

This notebook is a thesis-facing companion to the KLM Fig. 3 diagnostic.  It
keeps the side-by-side kappa panels, but replaces the continuous paper a-grid
with the thesis A-E regime ladder and fits rates for the four benchmark
schemes:

    FTE, ProjEuler, KL, KLM.

The regime ladder is defined by

    a = sigma^2 / (2 kappa theta).

For kappa=2 and theta=0.02, the A-E a-values reproduce the usual thesis
sigmas 0.10, 0.20, sqrt(0.08), 0.50, 0.80.  For kappa=0.2, sigma is rescaled
from the same a-values, so A-E represent the same regime severity in both
panels.

Full mode streams a HH reference with h_ref = 2^-22 by default.  The code never
materializes a (paths, 2^22) Brownian array; it processes path batches and fine
time chunks.
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import time
from datetime import datetime, timezone
from functools import partial
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")
os.environ.setdefault("JAX_ENABLE_X64", "true")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

PREFIX = "FOUR_SCHEME_REGIME_RATE"
RUN_MODE = os.environ.get(f"{PREFIX}_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError(f"{PREFIX}_RUN_MODE must be 'full' or 'smoke'")

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORKING_ROOT / "four_scheme_regime_rates_jax_outputs"
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
BACKUP_DIR = OUT_DIR / "backups"
LATEST_PARTIAL_BACKUP = BACKUP_DIR / "latest_partial.zip"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = int(os.environ.get(f"{PREFIX}_SEED", "120339106"))
FINAL_TIME = float(os.environ.get(f"{PREFIX}_T", "1.0"))
THETA = float(os.environ.get(f"{PREFIX}_THETA", "0.02"))
X0 = float(os.environ.get(f"{PREFIX}_X0", "0.02"))
ANCHOR_KAPPA = 2.0
ANCHOR_SIGMAS = {
    "A": 0.10,
    "B": 0.20,
    "C": 0.282842712474619,
    "D": 0.50,
    "E": 0.80,
}
REGIME_A_VALUES = {
    name: sigma * sigma / (2.0 * ANCHOR_KAPPA * THETA)
    for name, sigma in ANCHOR_SIGMAS.items()
}
SCHEMES = ["FTE", "ProjEuler", "KL", "KLM"]


def parse_int_list(value: str) -> list[int]:
    return [int(part.strip()) for part in value.split(",") if part.strip()]


def parse_float_list(value: str) -> list[float]:
    return [float(part.strip()) for part in value.split(",") if part.strip()]


def parse_str_list(value: str) -> list[str]:
    return [part.strip() for part in value.split(",") if part.strip()]


def reference_grid_from_env(default_power: int) -> tuple[int, int]:
    n_steps_key = f"{PREFIX}_REFERENCE_N_STEPS"
    power_key = f"{PREFIX}_REFERENCE_POWER"
    if n_steps_key in os.environ and power_key not in os.environ:
        n_steps = int(os.environ[n_steps_key])
        if n_steps <= 0 or n_steps & (n_steps - 1):
            raise ValueError(f"{n_steps_key} must be a positive power of two")
        return int(np.log2(n_steps)), n_steps
    reference_power = int(os.environ.get(power_key, str(default_power)))
    if reference_power <= 0:
        raise ValueError(f"{power_key} must be positive")
    return reference_power, 2**reference_power


if RUN_MODE == "smoke":
    N_PATHS = int(os.environ.get(f"{PREFIX}_N_PATHS", "64"))
    REFERENCE_POWER, REFERENCE_N_STEPS = reference_grid_from_env(default_power=9)
    LEVELS = parse_int_list(os.environ.get(f"{PREFIX}_LEVELS", "3,4,5"))
    PATH_BATCH_SIZE = int(os.environ.get(f"{PREFIX}_PATH_BATCH_SIZE", str(min(N_PATHS, 32))))
    CHUNK_STEPS = int(os.environ.get(f"{PREFIX}_CHUNK_STEPS", "128"))
else:
    N_PATHS = int(os.environ.get(f"{PREFIX}_N_PATHS", "1024"))
    REFERENCE_POWER, REFERENCE_N_STEPS = reference_grid_from_env(default_power=22)
    LEVELS = parse_int_list(os.environ.get(f"{PREFIX}_LEVELS", "3,4,5,6,7,8"))
    PATH_BATCH_SIZE = int(os.environ.get(f"{PREFIX}_PATH_BATCH_SIZE", "256"))
    CHUNK_STEPS = int(os.environ.get(f"{PREFIX}_CHUNK_STEPS", str(2**14)))

KAPPAS = parse_float_list(os.environ.get(f"{PREFIX}_KAPPAS", "2.0,0.2"))
REGIME_LIST = parse_str_list(os.environ.get(f"{PREFIX}_REGIMES", "A,B,C,D,E"))
COARSE_N_STEPS = [2**level for level in LEVELS]

if any(name not in REGIME_A_VALUES for name in REGIME_LIST):
    raise ValueError(f"{PREFIX}_REGIMES can only contain A,B,C,D,E")
if any(REFERENCE_N_STEPS % n_steps != 0 for n_steps in COARSE_N_STEPS):
    raise ValueError("Every coarse step count must divide the HH reference step count")
if REFERENCE_N_STEPS % CHUNK_STEPS != 0:
    raise ValueError(f"{PREFIX}_CHUNK_STEPS must divide the HH reference step count")
if PATH_BATCH_SIZE <= 0 or CHUNK_STEPS <= 0:
    raise ValueError(f"{PREFIX}_PATH_BATCH_SIZE and {PREFIX}_CHUNK_STEPS must be positive")

RESUME_ENV = "FOUR_SCHEME_REGIME_RATE_RESUME"
RESUME = os.environ.get(RESUME_ENV, "1").strip() not in {"0", "false", "False", "no", "NO"}
CHECKPOINT_EVERY_BATCHES = int(os.environ.get(f"{PREFIX}_CHECKPOINT_EVERY_BATCHES", "1"))
BACKUP_EVERY_BLOCKS = int(os.environ.get(f"{PREFIX}_BACKUP_EVERY_BLOCKS", "1"))
BATCH_CSV = RES_DIR / "four_scheme_regime_rate_batch_contributions.csv"
PARTIAL_DIAGNOSTICS_CSV = RES_DIR / "four_scheme_regime_rate_diagnostics_partial.csv"
PARTIAL_RATES_CSV = RES_DIR / "four_scheme_regime_rates_partial.csv"

RUN_CONFIG = {
    "run_mode": RUN_MODE,
    "n_paths": N_PATHS,
    "reference": "HH",
    "reference_power": REFERENCE_POWER,
    "reference_n_steps": REFERENCE_N_STEPS,
    "reference_dt": FINAL_TIME / REFERENCE_N_STEPS,
    "levels": LEVELS,
    "coarse_n_steps": COARSE_N_STEPS,
    "kappas": KAPPAS,
    "theta": THETA,
    "x0": X0,
    "regimes": REGIME_LIST,
    "regime_a_values": {name: REGIME_A_VALUES[name] for name in REGIME_LIST},
    "path_batch_size": PATH_BATCH_SIZE,
    "chunk_steps": CHUNK_STEPS,
    "schemes": SCHEMES,
    "resume": RESUME,
    "checkpoint_every_batches": CHECKPOINT_EVERY_BATCHES,
    "backup_every_blocks": BACKUP_EVERY_BLOCKS,
    "batch_contribution_csv": str(BATCH_CSV),
    "partial_diagnostics_csv": str(PARTIAL_DIAGNOSTICS_CSV),
    "regime_definition": "same a = sigma^2/(2 kappa theta) ladder in both kappa panels",
}
RUN_CONFIG_HASH_PAYLOAD = json.dumps(
    {
        key: value
        for key, value in RUN_CONFIG.items()
        if key not in {"resume", "checkpoint_every_batches", "backup_every_blocks", "run_config_hash"}
    },
    sort_keys=True,
    separators=(",", ":"),
)
RUN_CONFIG_HASH = hashlib.sha256(RUN_CONFIG_HASH_PAYLOAD.encode("utf-8")).hexdigest()[:16]
RUN_CONFIG["run_config_hash"] = RUN_CONFIG_HASH


def sigma_from_regime(kappa: float, regime_name: str) -> float:
    return float(np.sqrt(2.0 * kappa * THETA * REGIME_A_VALUES[regime_name]))



def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    tmp_path.replace(path)


def append_csv_rows(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    with path.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


def read_csv_rows(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


def utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def run_hash_payload(config: dict) -> dict:
    ignored = {"resume", "checkpoint_every_batches", "backup_every_blocks", "run_config_hash"}
    return {key: value for key, value in config.items() if key not in ignored}


def make_run_config_hash(config: dict) -> str:
    payload = json.dumps(run_hash_payload(config), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def batch_row_uid(row: dict) -> tuple:
    return (
        row.get("run_config_hash", ""),
        row.get("kappa", ""),
        row.get("regime", ""),
        row.get("batch_start", ""),
        row.get("scheme", ""),
        row.get("level", ""),
    )


def current_batch_rows(path: Path) -> list[dict]:
    unique = {}
    for row in read_csv_rows(path):
        if row.get("run_config_hash") == RUN_CONFIG_HASH:
            unique[batch_row_uid(row)] = row
    return list(unique.values())


def completed_batch_keys(path: Path) -> set[tuple[str, str, str]]:
    counts = {}
    expected = len(SCHEMES) * len(COARSE_N_STEPS)
    for row in current_batch_rows(path):
        key = (row["kappa"], row["regime"], row["batch_start"])
        counts[key] = counts.get(key, 0) + 1
    return {key for key, count in counts.items() if count >= expected}


def aggregate_batch_rows(batch_rows: list[dict]) -> list[dict]:
    groups = {}
    for row in batch_rows:
        key = (row["kappa"], row["regime"], row["scheme"], row["level"])
        if key not in groups:
            groups[key] = {
                "rows": [],
                "sum_abs": 0.0,
                "sum_sq": 0.0,
                "paths": 0,
                "step_count_sum": 0.0,
                "backstop_count": 0.0,
                "runtime_s": 0.0,
            }
        group = groups[key]
        group["rows"].append(row)
        group["sum_abs"] += float(row["sum_abs"])
        group["sum_sq"] += float(row["sum_sq"])
        group["paths"] += int(row["batch_size"])
        group["step_count_sum"] += float(row["step_count_sum"])
        group["backstop_count"] += float(row["backstop_count"])
        group["runtime_s"] += float(row["runtime_s"])

    regime_order = {name: idx for idx, name in enumerate(REGIME_LIST)}
    scheme_order = {name: idx for idx, name in enumerate(SCHEMES)}
    diagnostics = []
    for key in sorted(groups, key=lambda item: (float(item[0]), regime_order[item[1]], scheme_order[item[2]], int(item[3]))):
        first = groups[key]["rows"][0]
        paths = groups[key]["paths"]
        step_count_sum = groups[key]["step_count_sum"]
        diagnostics.append({
            "run_config_hash": RUN_CONFIG_HASH,
            "run_mode": RUN_MODE,
            "kappa": float(first["kappa"]),
            "theta": THETA,
            "x0": X0,
            "regime": first["regime"],
            "regime_a": float(first["regime_a"]),
            "sigma": float(first["sigma"]),
            "alpha": float(first["alpha"]),
            "scheme": first["scheme"],
            "scheme_variant": first["scheme_variant"],
            "reference": "HH",
            "reference_power": REFERENCE_POWER,
            "reference_n_steps": REFERENCE_N_STEPS,
            "reference_dt": FINAL_TIME / REFERENCE_N_STEPS,
            "n_paths": N_PATHS,
            "completed_paths": paths,
            "path_batch_size": PATH_BATCH_SIZE,
            "chunk_steps": CHUNK_STEPS,
            "level": int(first["level"]),
            "n_steps": int(first["n_steps"]),
            "dt": float(first["dt"]),
            "l1": groups[key]["sum_abs"] / paths,
            "l2": float(np.sqrt(groups[key]["sum_sq"] / paths)),
            "runtime_s": groups[key]["runtime_s"],
            "timing_scope": "batch-checkpointed-kappa-regime",
            "mean_steps_per_path": step_count_sum / paths,
            "backstop_fraction": groups[key]["backstop_count"] / max(step_count_sum, 1.0),
        })
    return diagnostics


def write_progress_manifest(stage: str, batch_rows: list[dict], diagnostics: list[dict]) -> None:
    expected_batches = len(KAPPAS) * len(REGIME_LIST) * len(range(0, N_PATHS, PATH_BATCH_SIZE))
    completed_batches = len({
        (row["kappa"], row["regime"], row["batch_start"])
        for row in batch_rows
    })
    manifest = {
        "stage": stage,
        "updated_utc": utc_timestamp(),
        "run_config_hash": RUN_CONFIG_HASH,
        "batch_contribution_rows": len(batch_rows),
        "diagnostic_rows": len(diagnostics),
        "completed_batches": completed_batches,
        "expected_batches": expected_batches,
        "completed_batch_fraction": completed_batches / expected_batches if expected_batches else 0.0,
        "batch_contribution_csv": str(BATCH_CSV),
        "partial_diagnostics_csv": str(PARTIAL_DIAGNOSTICS_CSV),
    }
    with (RES_DIR / "progress_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)


def refresh_outputs(final: bool = False) -> tuple[list[dict], list[dict]]:
    batch_rows = current_batch_rows(BATCH_CSV)
    diagnostics = aggregate_batch_rows(batch_rows)
    if diagnostics:
        write_csv(PARTIAL_DIAGNOSTICS_CSV, diagnostics)
        write_csv(RES_DIR / "four_scheme_regime_rate_diagnostics.csv", diagnostics)
    rates = compute_rates(diagnostics) if diagnostics else []
    if rates:
        write_csv(PARTIAL_RATES_CSV, rates)
        if final:
            write_csv(RES_DIR / "four_scheme_regime_rates.csv", rates)
    write_progress_manifest("final" if final else "partial", batch_rows, diagnostics)
    return diagnostics, rates


def fit_loglog_order(step_sizes, errors):
    h = np.asarray(step_sizes, dtype=float)
    e = np.maximum(np.asarray(errors, dtype=float), 1e-300)
    return float(np.polyfit(np.log(h), np.log(e), 1)[0])


def kl_alpha(kappa, theta, sigma):
    return (4.0 * kappa * theta - sigma**2) / 8.0


def hh_step(x, kappa, theta, sigma, dt, dW):
    floor = 0.25 * sigma**2 * dt
    r1 = jnp.maximum(
        0.5 * sigma * jnp.sqrt(dt),
        jnp.sqrt(jnp.maximum(floor, x)) + 0.5 * sigma * dW,
    )
    x_hat = r1 * r1 + dt * (kappa * (theta - x) - 0.25 * sigma**2)
    return jnp.maximum(x_hat, 0.0)


def implicit_step(y, h, dW, alpha, beta, gamma):
    a = 1.0 - beta * h
    u = y + gamma * dW
    return (u + jnp.sqrt(u * u + 4.0 * a * alpha * h)) / (2.0 * a)


def projected_backstop_step(y, h, dW, alpha, beta, gamma):
    y_floor = gamma * jnp.sqrt(h)
    y_hat = y + (alpha / y + beta * y) * h + gamma * dW
    return jnp.maximum(y_hat, y_floor)


def choose_kl_adaptive_interval(x, remaining_steps, kappa, theta, sigma, h_values, dt_fine, rho=2.0, safety=0.95):
    alpha = kl_alpha(kappa, theta, sigma)
    x_zero = theta * (1.0 - jnp.exp(-kappa * h_values)) / rho
    in_soft_zero = x < x_zero
    x_for_soft = jnp.minimum(x, x_zero * 0.999999)
    dt_soft = -jnp.log((x_zero - theta) / (x_for_soft - theta)) / kappa
    dt_adaptive = safety * x / (2.0 * jnp.abs(alpha))
    split_h = jnp.minimum(jnp.minimum(dt_adaptive, h_values), remaining_steps * dt_fine)
    h_cont = jnp.where(in_soft_zero, jnp.minimum(dt_soft, remaining_steps * dt_fine), split_h)
    m = jnp.floor(h_cont / dt_fine).astype(jnp.int64)
    m = jnp.where(remaining_steps > 0, jnp.maximum(m, 1), 0)
    m = jnp.minimum(m, remaining_steps)
    return m, jnp.where(remaining_steps > 0, in_soft_zero, False)


def choose_klm_interval(y, remaining_steps, h_values, dt_fine, rho=64.0):
    h_min = h_values / rho
    m_min = jnp.maximum(jnp.rint(h_min / dt_fine).astype(jnp.int64), 1)
    m_max = jnp.maximum(jnp.floor(h_values / dt_fine).astype(jnp.int64), 1)
    h_prop = h_values * jnp.minimum(1.0, jnp.abs(y))
    m_raw = jnp.floor(h_prop / dt_fine).astype(jnp.int64)
    min_triggered = m_raw < m_min
    m = jnp.where(min_triggered, m_min, jnp.minimum(m_raw, m_max))
    m = jnp.where(remaining_steps > 0, jnp.minimum(m, remaining_steps), 0)
    return m, jnp.where(remaining_steps > 0, min_triggered, False)


def initial_stream_state(batch_size, kappa, theta, sigma, factors, h_values, use_kl_adaptive):
    n_levels = len(COARSE_N_STEPS)
    factors = jnp.asarray(factors, dtype=jnp.int64).reshape((n_levels, 1))
    h_values = jnp.asarray(h_values, dtype=jnp.float64).reshape((n_levels, 1))
    level_shape = (n_levels, batch_size)
    reference = jnp.full((batch_size,), X0, dtype=jnp.float64)
    fixed_acc = jnp.zeros(level_shape, dtype=jnp.float64)
    fixed_remaining = jnp.broadcast_to(factors, level_shape)
    fte_aux = jnp.full(level_shape, X0, dtype=jnp.float64)
    proj_y = jnp.maximum(jnp.sqrt(X0), 0.5 * sigma * jnp.sqrt(h_values)) * jnp.ones(level_shape, dtype=jnp.float64)
    kl_x = jnp.full(level_shape, X0, dtype=jnp.float64)
    kl_acc = jnp.zeros(level_shape, dtype=jnp.float64)
    if use_kl_adaptive:
        kl_m, kl_soft = choose_kl_adaptive_interval(
            kl_x, REFERENCE_N_STEPS, kappa, theta, sigma, h_values, FINAL_TIME / REFERENCE_N_STEPS
        )
    else:
        kl_m = jnp.zeros(level_shape, dtype=jnp.int64)
        kl_soft = jnp.zeros(level_shape, dtype=bool)
    kl_remaining = kl_m
    kl_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    kl_soft_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    klm_y = jnp.full(level_shape, jnp.sqrt(X0), dtype=jnp.float64)
    klm_acc = jnp.zeros(level_shape, dtype=jnp.float64)
    klm_m, klm_min = choose_klm_interval(
        klm_y, REFERENCE_N_STEPS, h_values, FINAL_TIME / REFERENCE_N_STEPS
    )
    klm_remaining = klm_m
    klm_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    klm_min_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    klm_neg_total = jnp.zeros((n_levels,), dtype=jnp.int64)
    return (
        reference,
        fixed_acc,
        fixed_remaining,
        fte_aux,
        proj_y,
        kl_x,
        kl_acc,
        kl_remaining,
        kl_m,
        kl_soft,
        kl_total,
        kl_soft_total,
        klm_y,
        klm_acc,
        klm_remaining,
        klm_m,
        klm_min,
        klm_total,
        klm_min_total,
        klm_neg_total,
    )


def make_dW_chunk(kappa_index, regime_index, batch_start, chunk_start, batch_size, chunk_steps, dt_fine):
    key = jax.random.PRNGKey(MASTER_SEED)
    key = jax.random.fold_in(key, int(kappa_index))
    key = jax.random.fold_in(key, int(regime_index))
    key = jax.random.fold_in(key, int(batch_start))
    key = jax.random.fold_in(key, int(chunk_start))
    return jax.random.normal(key, (batch_size, chunk_steps), dtype=jnp.float64) * jnp.sqrt(dt_fine)


@partial(jax.jit, static_argnames=("use_kl_adaptive",))
def stream_strong_chunk(
    state,
    dW_chunk,
    start_index,
    reference_n_steps,
    dt_fine,
    kappa,
    theta,
    sigma,
    factors,
    h_values,
    use_kl_adaptive,
):
    factors = factors.astype(jnp.int64)
    h_values = h_values.astype(jnp.float64)
    alpha = kl_alpha(kappa, theta, sigma)

    def one_fine_step(carry, inputs):
        dW_col, fine_index = inputs
        remaining_after = reference_n_steps - fine_index - 1
        (
            reference,
            fixed_acc,
            fixed_remaining,
            fte_aux,
            proj_y,
            kl_x,
            kl_acc,
            kl_remaining,
            kl_m,
            kl_soft,
            kl_total,
            kl_soft_total,
            klm_y,
            klm_acc,
            klm_remaining,
            klm_m,
            klm_min,
            klm_total,
            klm_min_total,
            klm_neg_total,
        ) = carry

        reference = hh_step(reference, kappa, theta, sigma, dt_fine, dW_col)

        fixed_acc_next = fixed_acc + dW_col[None, :]
        fixed_remaining_next = fixed_remaining - 1
        fixed_land = fixed_remaining_next <= 0

        fte_pos = jnp.maximum(fte_aux, 0.0)
        fte_candidate = fte_aux + kappa * (theta - fte_pos) * h_values + sigma * jnp.sqrt(fte_pos) * fixed_acc_next
        fte_aux = jnp.where(fixed_land, fte_candidate, fte_aux)

        y_floor = 0.5 * sigma * jnp.sqrt(h_values)
        proj_safe = jnp.maximum(proj_y, y_floor)
        proj_candidate = proj_safe + (alpha / proj_safe - 0.5 * kappa * proj_safe) * h_values + 0.5 * sigma * fixed_acc_next
        proj_y = jnp.where(fixed_land, jnp.maximum(proj_candidate, y_floor), proj_y)

        if use_kl_adaptive:
            kl_acc_next = kl_acc + dW_col[None, :]
            kl_remaining_next = kl_remaining - 1
            kl_land = kl_remaining_next <= 0
            kl_h = kl_m * dt_fine
            kl_decay = jnp.exp(-kappa * kl_h)
            kl_soft_candidate = kl_decay * kl_x + theta * (1.0 - kl_decay)
            kl_inside = jnp.maximum(kl_x + 2.0 * alpha * kl_h, 0.0)
            kl_split_candidate = kl_decay * (jnp.sqrt(kl_inside) + 0.5 * sigma * kl_acc_next) ** 2
            kl_candidate = jnp.where(kl_soft, kl_soft_candidate, kl_split_candidate)
            kl_x = jnp.where(kl_land, kl_candidate, kl_x)
            new_kl_m, new_kl_soft = choose_kl_adaptive_interval(
                kl_x, remaining_after, kappa, theta, sigma, h_values, dt_fine
            )
            kl_total = kl_total + jnp.sum(kl_land, axis=1)
            kl_soft_total = kl_soft_total + jnp.sum(kl_land & kl_soft, axis=1)
            kl_acc = jnp.where(kl_land, 0.0, kl_acc_next)
            kl_remaining = jnp.where(kl_land, new_kl_m, kl_remaining_next)
            kl_m = jnp.where(kl_land, new_kl_m, kl_m)
            kl_soft = jnp.where(kl_land, new_kl_soft, kl_soft)
        else:
            kl_decay = jnp.exp(-kappa * h_values)
            kl_inside = jnp.maximum(kl_x + 2.0 * alpha * h_values, 0.0)
            kl_candidate = kl_decay * (jnp.sqrt(kl_inside) + 0.5 * sigma * fixed_acc_next) ** 2
            kl_x = jnp.where(fixed_land, kl_candidate, kl_x)

        fixed_acc = jnp.where(fixed_land, 0.0, fixed_acc_next)
        fixed_remaining = jnp.where(fixed_land, factors, fixed_remaining_next)

        klm_acc_next = klm_acc + dW_col[None, :]
        klm_remaining_next = klm_remaining - 1
        klm_land = klm_remaining_next <= 0
        klm_h = klm_m * dt_fine
        beta = -0.5 * kappa
        gamma = 0.5 * sigma
        klm_explicit = jnp.where(
            klm_h > 0.0,
            klm_y + (alpha / klm_y + beta * klm_y) * klm_h + gamma * klm_acc_next,
            klm_y,
        )
        klm_implicit = implicit_step(klm_y, klm_h, klm_acc_next, alpha, beta, gamma)
        klm_projected = projected_backstop_step(klm_y, klm_h, klm_acc_next, alpha, beta, gamma)
        klm_backstop = jnp.where(alpha > 0.0, klm_implicit, klm_projected)
        klm_neg = (~klm_min) & (klm_explicit <= 0.0)
        klm_use_backstop = klm_min | klm_neg
        klm_candidate = jnp.where(klm_use_backstop, klm_backstop, klm_explicit)
        klm_y = jnp.where(klm_land, klm_candidate, klm_y)
        new_klm_m, new_klm_min = choose_klm_interval(klm_y, remaining_after, h_values, dt_fine)
        klm_total = klm_total + jnp.sum(klm_land, axis=1)
        klm_min_total = klm_min_total + jnp.sum(klm_land & klm_min, axis=1)
        klm_neg_total = klm_neg_total + jnp.sum(klm_land & klm_neg, axis=1)
        klm_acc = jnp.where(klm_land, 0.0, klm_acc_next)
        klm_remaining = jnp.where(klm_land, new_klm_m, klm_remaining_next)
        klm_m = jnp.where(klm_land, new_klm_m, klm_m)
        klm_min = jnp.where(klm_land, new_klm_min, klm_min)

        return (
            reference,
            fixed_acc,
            fixed_remaining,
            fte_aux,
            proj_y,
            kl_x,
            kl_acc,
            kl_remaining,
            kl_m,
            kl_soft,
            kl_total,
            kl_soft_total,
            klm_y,
            klm_acc,
            klm_remaining,
            klm_m,
            klm_min,
            klm_total,
            klm_min_total,
            klm_neg_total,
        ), None

    fine_indices = start_index + jnp.arange(dW_chunk.shape[1], dtype=jnp.int64)
    new_state, _ = jax.lax.scan(one_fine_step, state, (dW_chunk.T, fine_indices))
    return new_state



def run_one_kappa_regime(kappa_index: int, kappa: float, regime_index: int, regime_name: str) -> list[dict]:
    sigma = sigma_from_regime(kappa, regime_name)
    a_value = REGIME_A_VALUES[regime_name]
    alpha = kl_alpha(kappa, THETA, sigma)
    use_kl_adaptive = alpha < 0.0
    dt_fine = FINAL_TIME / REFERENCE_N_STEPS
    factors_np = np.asarray([REFERENCE_N_STEPS // n_steps for n_steps in COARSE_N_STEPS], dtype=np.int64)
    h_values_np = np.asarray([FINAL_TIME / n_steps for n_steps in COARSE_N_STEPS], dtype=float)
    factors = jnp.asarray(factors_np.reshape((-1, 1)), dtype=jnp.int64)
    h_values = jnp.asarray(h_values_np.reshape((-1, 1)), dtype=jnp.float64)
    completed = completed_batch_keys(BATCH_CSV) if RESUME else set()
    block_start = time.perf_counter()

    for batch_start in range(0, N_PATHS, PATH_BATCH_SIZE):
        batch_key = (str(float(kappa)), regime_name, str(batch_start))
        if batch_key in completed:
            print(f"  skipping completed batch kappa={kappa:g}, regime={regime_name}, batch_start={batch_start}")
            continue

        batch_runtime_start = time.perf_counter()
        batch_size = min(PATH_BATCH_SIZE, N_PATHS - batch_start)
        state = initial_stream_state(
            batch_size,
            kappa,
            THETA,
            sigma,
            factors_np,
            h_values_np,
            use_kl_adaptive,
        )
        for chunk_start in range(0, REFERENCE_N_STEPS, CHUNK_STEPS):
            dW_chunk = make_dW_chunk(kappa_index, regime_index, batch_start, chunk_start, batch_size, CHUNK_STEPS, dt_fine)
            state = stream_strong_chunk(
                state,
                dW_chunk,
                jnp.asarray(chunk_start, dtype=jnp.int64),
                jnp.asarray(REFERENCE_N_STEPS, dtype=jnp.int64),
                jnp.asarray(dt_fine, dtype=jnp.float64),
                jnp.asarray(kappa, dtype=jnp.float64),
                jnp.asarray(THETA, dtype=jnp.float64),
                jnp.asarray(sigma, dtype=jnp.float64),
                factors,
                h_values,
                use_kl_adaptive=use_kl_adaptive,
            )
        state = jax.block_until_ready(state)
        (
            reference,
            _fixed_acc,
            _fixed_remaining,
            fte_aux,
            proj_y,
            kl_x,
            _kl_acc,
            _kl_remaining,
            _kl_m,
            _kl_soft,
            kl_total_batch,
            kl_soft_batch,
            klm_y,
            _klm_acc,
            _klm_remaining,
            _klm_m,
            _klm_min,
            klm_total_batch,
            klm_min_batch,
            klm_neg_batch,
        ) = state
        reference_np = np.asarray(reference)
        terminals = {
            "FTE": np.maximum(np.asarray(fte_aux), 0.0),
            "ProjEuler": np.asarray(proj_y) ** 2,
            "KL": np.asarray(kl_x),
            "KLM": np.asarray(klm_y) ** 2,
        }
        batch_runtime = time.perf_counter() - batch_runtime_start
        timestamp = utc_timestamp()
        batch_rows = []
        for scheme in SCHEMES:
            diff = terminals[scheme] - reference_np[None, :]
            sum_abs = np.sum(np.abs(diff), axis=1)
            sum_sq = np.sum(diff * diff, axis=1)
            for idx, n_steps in enumerate(COARSE_N_STEPS):
                if scheme == "KL" and use_kl_adaptive:
                    step_count_sum = float(np.asarray(kl_total_batch)[idx])
                    backstop_count = float(np.asarray(kl_soft_batch)[idx])
                    scheme_variant = "adaptive-soft-zero"
                elif scheme == "KLM":
                    step_count_sum = float(np.asarray(klm_total_batch)[idx])
                    backstop_count = float((np.asarray(klm_min_batch) + np.asarray(klm_neg_batch))[idx])
                    scheme_variant = "adaptive-backstop"
                else:
                    step_count_sum = float(n_steps * batch_size)
                    backstop_count = 0.0
                    scheme_variant = "uniform" if scheme == "KL" else "fixed"
                batch_rows.append({
                    "run_config_hash": RUN_CONFIG_HASH,
                    "timestamp_utc": timestamp,
                    "run_mode": RUN_MODE,
                    "kappa": float(kappa),
                    "theta": THETA,
                    "x0": X0,
                    "regime": regime_name,
                    "regime_a": a_value,
                    "sigma": sigma,
                    "alpha": alpha,
                    "scheme": scheme,
                    "scheme_variant": scheme_variant,
                    "reference": "HH",
                    "reference_power": REFERENCE_POWER,
                    "reference_n_steps": REFERENCE_N_STEPS,
                    "reference_dt": dt_fine,
                    "n_paths": N_PATHS,
                    "path_batch_size": PATH_BATCH_SIZE,
                    "chunk_steps": CHUNK_STEPS,
                    "level": LEVELS[idx],
                    "n_steps": n_steps,
                    "dt": FINAL_TIME / n_steps,
                    "batch_start": batch_start,
                    "batch_size": batch_size,
                    "sum_abs": float(sum_abs[idx]),
                    "sum_sq": float(sum_sq[idx]),
                    "runtime_s": batch_runtime,
                    "timing_scope": "single-path-batch",
                    "step_count_sum": step_count_sum,
                    "backstop_count": backstop_count,
                })
        append_csv_rows(BATCH_CSV, batch_rows)
        print(
            f"  checkpointed batch kappa={kappa:g}, regime={regime_name}, "
            f"batch_start={batch_start}, paths={batch_size}, runtime={batch_runtime:.1f}s"
        )
        refresh_outputs(final=False)

    diagnostics, _rates = refresh_outputs(final=False)
    print(
        f"completed block kappa={kappa:g}, regime={regime_name} in "
        f"{time.perf_counter() - block_start:.1f}s"
    )
    return [
        row for row in diagnostics
        if float(row["kappa"]) == float(kappa) and row["regime"] == regime_name
    ]


def compute_rates(error_rows: list[dict]) -> list[dict]:
    rate_rows = []
    keys = sorted({
        (row["kappa"], row["regime"], row["scheme"])
        for row in error_rows
    })
    for kappa, regime, scheme in keys:
        sr = [row for row in error_rows if row["kappa"] == kappa and row["regime"] == regime and row["scheme"] == scheme]
        sr = sorted(sr, key=lambda row: row["dt"])
        first = sr[0]
        variants = sorted({row["scheme_variant"] for row in sr})
        rate_rows.append({
            "run_mode": RUN_MODE,
            "kappa": kappa,
            "theta": THETA,
            "x0": X0,
            "regime": regime,
            "regime_a": first["regime_a"],
            "sigma": first["sigma"],
            "alpha": first["alpha"],
            "scheme": scheme,
            "scheme_variant": "+".join(variants),
            "reference": "HH",
            "reference_power": REFERENCE_POWER,
            "reference_n_steps": REFERENCE_N_STEPS,
            "n_paths": N_PATHS,
            "levels": ",".join(str(level) for level in LEVELS),
            "fitted_order_l1": fit_loglog_order([row["dt"] for row in sr], [row["l1"] for row in sr]),
            "fitted_order_l2": fit_loglog_order([row["dt"] for row in sr], [row["l2"] for row in sr]),
            "finest_dt": min(row["dt"] for row in sr),
            "finest_l2": sr[0]["l2"],
            "coarsest_dt": max(row["dt"] for row in sr),
            "coarsest_l2": sr[-1]["l2"],
        })
    return rate_rows



def plot_rate_panels(rate_rows: list[dict]) -> None:
    if not rate_rows:
        return
    style = {
        "FTE": dict(color="#1f77b4", marker="o", ls="-"),
        "ProjEuler": dict(color="#2ca02c", marker="^", ls="-"),
        "KL": dict(color="#d62728", marker="D", ls="-"),
        "KLM": dict(color="#9467bd", marker="v", ls="-"),
    }
    fig, axes = plt.subplots(1, len(KAPPAS), figsize=(11.0, 4.5), sharey=True)
    if len(KAPPAS) == 1:
        axes = [axes]
    x = np.arange(len(REGIME_LIST))
    labels = [f"{name}\na={REGIME_A_VALUES[name]:g}" for name in REGIME_LIST]
    for ax, kappa in zip(axes, KAPPAS):
        panel_rows = [row for row in rate_rows if float(row["kappa"]) == float(kappa)]
        for scheme in SCHEMES:
            y = []
            for regime in REGIME_LIST:
                row = next((item for item in panel_rows if item["regime"] == regime and item["scheme"] == scheme), None)
                y.append(np.nan if row is None else float(row["fitted_order_l2"]))
            if np.isfinite(y).any():
                ax.plot(x, y, label=scheme, **style[scheme])
        ax.axhline(0.5, color="0.25", lw=0.9, ls="--")
        ax.axhline(1.0, color="0.25", lw=0.9, ls=":")
        ax.axvline(2.5, color="0.65", lw=0.9, ls="--")
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_xlabel("regime ladder")
        ax.set_title(f"kappa = {kappa:g}")
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.25, 1.35)
    axes[0].set_ylabel("fitted strong L2 order")
    axes[-1].legend(loc="upper right", fontsize=8)
    fig.suptitle(f"Four-scheme regime-rate sweep, HH h_ref=2^-{REFERENCE_POWER}")
    fig.tight_layout()
    png_path = FIG_DIR / "four_scheme_regime_rates_jax.png"
    pdf_path = FIG_DIR / "four_scheme_regime_rates_jax.pdf"
    fig.savefig(png_path, dpi=220)
    fig.savefig(pdf_path)
    plt.close(fig)
    print("saved", png_path)
    print("saved", pdf_path)


def plot_error_panels(error_rows: list[dict]) -> None:
    for kappa in KAPPAS:
        fig, axes = plt.subplots(1, len(REGIME_LIST), figsize=(15.0, 3.3), sharey=False)
        if len(REGIME_LIST) == 1:
            axes = [axes]
        for ax, regime in zip(axes, REGIME_LIST):
            rows = [row for row in error_rows if float(row["kappa"]) == float(kappa) and row["regime"] == regime]
            for scheme in SCHEMES:
                sr = sorted([row for row in rows if row["scheme"] == scheme], key=lambda row: row["dt"])
                ax.loglog([row["dt"] for row in sr], [row["l2"] for row in sr], "o-", label=scheme)
            ax.set_title(f"{regime}, a={REGIME_A_VALUES[regime]:g}")
            ax.set_xlabel("h")
            ax.grid(True, which="both", alpha=0.25)
        axes[0].set_ylabel("strong L2 error")
        axes[-1].legend(fontsize=7)
        fig.suptitle(f"kappa = {kappa:g}, HH h_ref=2^-{REFERENCE_POWER}")
        fig.tight_layout()
        path = FIG_DIR / f"four_scheme_errors_kappa_{str(kappa).replace('.', 'p')}.pdf"
        fig.savefig(path)
        plt.close(fig)
        print("saved", path)



def write_run_config() -> None:
    with (OUT_DIR / "run_config.json").open("w", encoding="utf-8") as file:
        json.dump(RUN_CONFIG, file, indent=2)


def backup_outputs(label: str = "latest_partial") -> None:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    tmp_base = WORKING_ROOT / f"{OUT_DIR.name}_{label}_backup_tmp"
    archive = shutil.make_archive(str(tmp_base), "zip", OUT_DIR)
    target = LATEST_PARTIAL_BACKUP if label == "latest_partial" else BACKUP_DIR / f"{label}.zip"
    if target.exists():
        target.unlink()
    shutil.move(archive, target)
    print(f"backup written to {target}")


def archive_outputs() -> None:
    archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print(f"archived outputs to {archive}")


def main() -> None:
    print("devices:", jax.devices())
    print(
        f"RUN_MODE={RUN_MODE}, M={N_PATHS}, h_ref=2^-{REFERENCE_POWER}, "
        f"N_REF={REFERENCE_N_STEPS:,}, batch={PATH_BATCH_SIZE}, chunk_steps={CHUNK_STEPS}"
    )
    print(f"resume={RESUME}, run_config_hash={RUN_CONFIG_HASH}")
    print(f"kappas={KAPPAS}, regimes={REGIME_LIST}, levels={LEVELS}")
    print("regime a-values:", {name: REGIME_A_VALUES[name] for name in REGIME_LIST})
    write_run_config()
    if not RESUME:
        for path in [
            BATCH_CSV,
            PARTIAL_DIAGNOSTICS_CSV,
            PARTIAL_RATES_CSV,
            RES_DIR / "four_scheme_regime_rate_diagnostics.csv",
            RES_DIR / "four_scheme_regime_rates.csv",
        ]:
            if path.exists():
                path.unlink()

    block_counter = 0
    for kappa_index, kappa in enumerate(KAPPAS):
        print(f"\n=== kappa={kappa:g} ===")
        for regime_index, regime_name in enumerate(REGIME_LIST):
            sigma = sigma_from_regime(kappa, regime_name)
            print(f"regime {regime_name}: a={REGIME_A_VALUES[regime_name]:g}, sigma={sigma:g}")
            run_one_kappa_regime(kappa_index, kappa, regime_index, regime_name)
            diagnostics, rates = refresh_outputs(final=False)
            if rates:
                plot_rate_panels(rates)
            if diagnostics:
                plot_error_panels(diagnostics)
            block_counter += 1
            if BACKUP_EVERY_BLOCKS > 0 and block_counter % BACKUP_EVERY_BLOCKS == 0:
                backup_outputs("latest_partial")

    diagnostics, rate_rows = refresh_outputs(final=True)
    if rate_rows:
        plot_rate_panels(rate_rows)
    if diagnostics:
        plot_error_panels(diagnostics)
    backup_outputs("latest_partial")
    archive_outputs()
    print("done")


main()
